### [CPI, Gold, and SP500: Building an Inflation Hedge Strategy](https://ai.gopubby.com/cpi-gold-and-sp500-building-an-inflation-hedge-strategy-in-python-7ea78f66d022)

> The **Consumer Price Index (CPI)** is an indicator used to measure the increase in the average price of goods and services, which affects our daily expenses.

In [ ]:
import requests
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import os, sys

import warnings
warnings.filterwarnings('ignore')

from google.colab import userdata

# Make sure you have added FMP_API_KEY to Colab's secrets
try:
  token = userdata.get('FMP_API_KEY')
except userdata.SecretNotFoundError as snfe:
  print(f"Error: {snfe}")
  token = os.environ.get('FMP_API_KEY')
except Exception as e:
  print(f"Error: {e}")
  sys.exit(1)

fromdate = '1975-01-01'
todate = '2025-10-02'

In [ ]:
url = f'https://financialmodelingprep.com/stable/economic-indicators'
name = 'CPI'
querystring = {"apikey": token, "name": name, "from": fromdate, "to": todate}

resp = requests.get(url, querystring).json()

df_cpi = pd.DataFrame(resp)
df_cpi.drop(columns=["name"], inplace=True)
df_cpi.tail(5)

In [ ]:
url = f'https://financialmodelingprep.com/stable/historical-price-eod/light'
symbol = 'GCUSD'

querystring = {"apikey": token, "symbol": symbol, "from": fromdate, "to": todate}
resp = requests.get(url, querystring).json()

df_gold = pd.DataFrame(resp)
df_gold.drop(columns=["symbol","volume"], inplace=True)
df_gold.tail(5)

In [ ]:
df_cpi['date'] = pd.to_datetime(df_cpi['date']).dt.normalize()
df_gold['date'] = pd.to_datetime(df_gold['date']).dt.normalize()

df_cpi_sorted = df_cpi.sort_values('date')
df_gold_sorted = df_gold.sort_values('date')

df_cpi_gold = pd.merge_asof(
    df_cpi_sorted,
    df_gold_sorted,
    on='date',
    direction='backward'
).reset_index(drop=True)

df_cpi_gold = df_cpi_gold.rename(columns={'value': 'cpi', 'price': 'gold'})
df_cpi_gold.tail(5)

In [ ]:
subset = df_cpi_gold[['cpi', 'gold']].dropna()
value_price_corr = subset['cpi'].corr(subset['gold'], method='pearson')
pd.DataFrame({'cpi_gold_pearson_correlation': [value_price_corr]})

In [ ]:
years = 10
window = years*12
df_cpi_gold['rolling_corr'] = df_cpi_gold['cpi'].rolling(window=window, min_periods=window).corr(df_cpi_gold['gold'])

plt.figure(figsize=(12, 4))
plot_data = df_cpi_gold[['date', 'rolling_corr']].dropna()
plt.plot(plot_data['date'], plot_data['rolling_corr'])
plt.title(f'Rolling {years}-Years Correlation: CPI Value vs Gold Price')
plt.xlabel('Date')
plt.ylabel('Correlation')
plt.ylim(-1, 1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
ticker = '^GSPC'
url = f'https://financialmodelingprep.com/api/v3/historical-price-full/{ticker}'
querystring = {"apikey": token, "from": fromdate, "to": todate}
data = requests.get(url, querystring)

if data.status_code != 200:
    print(f"Error: {data.status_code}")
    print(data.text)

data = data.json()

df_sp500 = pd.DataFrame(data['historical'])

df_cpi_gold['date'] = pd.to_datetime(df_cpi_gold['date']).dt.normalize()

sp = df_sp500[['date', 'adjClose']].copy()
sp['date'] = pd.to_datetime(sp['date']).dt.normalize()
sp = sp.sort_values('date').rename(columns={'adjClose': 'sp500'})

df_backtest = pd.merge_asof(
    df_cpi_gold.sort_values('date'),
    sp,
    on='date',
    direction='backward'
).reset_index(drop=True)

df_backtest = df_backtest[['date', 'cpi', 'gold', 'sp500']]
df_backtest.tail(5)

In [ ]:
df_backtest['cpi_pct_change'] = df_cpi_gold['cpi'].pct_change()
df_backtest['gold_pct_change'] = df_cpi_gold['gold'].pct_change()
df_backtest['sp500_pct_change'] = df_backtest['sp500'].pct_change()
df_backtest

In [ ]:
df_backtest.dropna(inplace=True)
df_backtest['quantile_threshold'] = df_backtest['cpi_pct_change'].rolling(window=1000, min_periods=1).quantile(0.8)
df_backtest['signal'] = (df_backtest['cpi_pct_change'] > df_backtest['quantile_threshold']).shift(3)

df_backtest['strategy_return'] = df_backtest['gold_pct_change'].where(df_backtest['signal'],
                                                                      df_backtest['sp500_pct_change'])
df_backtest

In [ ]:
df_backtest['gold_equity'] = (1 + df_backtest['gold_pct_change']).cumprod()
df_backtest['sp500_equity'] = (1 + df_backtest['sp500_pct_change']).cumprod()
df_backtest['strategy_equity'] = (1 + df_backtest['strategy_return']).cumprod()

df_backtest[["gold_equity", "sp500_equity", "strategy_equity"]].tail(1)

In [ ]:
required_cols = {'date', 'gold_equity', 'sp500_equity', 'strategy_equity'}
missing = required_cols - set(df_backtest.columns)
if missing:
    raise ValueError(f"Missing required columns in df_backtest: {missing}")

df_plot_eq = df_backtest.copy()
df_plot_eq['date'] = pd.to_datetime(df_plot_eq['date'])
df_plot_eq = df_plot_eq.sort_values('date')

plt.figure(figsize=(11, 6))
plt.plot(df_plot_eq['date'], df_plot_eq['gold_equity'], label='Gold Equity', linewidth=1.5)
plt.plot(df_plot_eq['date'], df_plot_eq['sp500_equity'], label='S&P 500 Equity', linewidth=1.5)
plt.plot(df_plot_eq['date'], df_plot_eq['strategy_equity'], label='Strategy Equity', linewidth=2.0)

plt.title('Equity Curves: Gold vs S&P 500 vs Strategy')
plt.xlabel('Date')
plt.ylabel('Equity (Cumulative, base=1.0)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def calculate_max_drawdown(returns):
    wealth_index = (1 + returns).cumprod()
    running_max = wealth_index.cummax()
    drawdown = (wealth_index - running_max) / running_max
    max_drawdown = drawdown.min()
    return max_drawdown

print(f'GOLD has maximum drawdown of {calculate_max_drawdown(df_backtest["gold_pct_change"])}')
print(f'SP500 has maximum drawdown of {calculate_max_drawdown(df_backtest["sp500_pct_change"])}')
print(f'Strategy has maximum drawdown of {calculate_max_drawdown(df_backtest["strategy_return"])}')

In [ ]:
def calculate_sharpe_ratio(returns, risk_free_rate, periods_per_year):
    excess_return = returns - risk_free_rate / periods_per_year
    mean_excess_return = excess_return.mean() * periods_per_year
    vol = excess_return.std() * (periods_per_year ** 0.5)
    sharpe_ratio = mean_excess_return / vol
    return sharpe_ratio

print(f'GOLD has Sharpe Ratio of {calculate_sharpe_ratio(df_backtest["gold_pct_change"], risk_free_rate=0.02, periods_per_year=12)}')
print(f'SP500 has Sharpe Ratio of {calculate_sharpe_ratio(df_backtest["sp500_pct_change"], risk_free_rate=0.02, periods_per_year=12)}')
print(f'Strategy has Sharpe Ratio of {calculate_sharpe_ratio(df_backtest["strategy_return"], risk_free_rate=0.02, periods_per_year=12)}')